## Setup

In [8]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

## HISTORICAL DATA LOADING AND PREPROCESSING

In [9]:
!python ../src/scripts/data_preprocessing.py

Processed dataset saved at data/processed/market_data.csv


In [10]:
file_path = "../data/processed/market_data.csv"
df_market = pd.read_csv(file_path)
print(df_market.head())
print(df_market.shape)

         Date  equity_return  debt_return  cash_return  volatility  drawdown  \
0  2007-01-01         0.0293       0.0441       0.0025    0.061391  0.000000   
1  2007-02-01        -0.0826       0.0074       0.0025    0.065611 -0.082641   
2  2007-03-01         0.0204       0.0182       0.0025    0.064439 -0.063965   
3  2007-04-01         0.0697       0.0180       0.0025    0.061713  0.000000   
4  2007-05-01         0.0509      -0.0048       0.0025    0.060363  0.000000   

   market_indicator  
0                 2  
1                 1  
2                 2  
3                 2  
4                 2  
(229, 7)


## SYNTHETIC USER PROFILES GENERATION

In [24]:
!python ../src/scripts/user_profiles.py

USer profiles saved at data/processed/user_profiles.csv


In [25]:
file_path = "../data/processed/user_profiles.csv"
df_user = pd.read_csv(file_path)
print(df_user.head())
print(df_user.shape)

    wealth  age  goal  income  income_var  job_loss_prob  expense_base  \
0    26674   28   456   40259        0.05          0.001         38121   
1  4489830   37   348  114438        0.05          0.001         61888   
2    82970   36   360   32669        0.15          0.001         30402   
3  4682792   54   144  120766        0.05          0.001         64439   
4   424614   36   360   34408        0.15          0.004         28752   

   emergency_buffer  emergency_buffer_target  risk_appetite  
0                 6                        6            0.3  
1                 4                        4            0.5  
2                 6                        6            0.3  
3                 5                        5            0.5  
4                 9                        9            0.5  
(20000, 10)


In [26]:
df_user.describe()

,wealth,age,goal,income,income_var,job_loss_prob,expense_base,emergency_buffer,emergency_buffer_target,risk_appetite
count,2.000000e+04,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,8.849675e+06,40.398550,307.217400,100458.465450,0.136715,0.002646,60964.693500,8.044550,8.044550,0.472940
std,1.963039e+07,10.958897,131.506758,74196.959021,0.085102,0.002136,37761.071427,4.012888,4.012888,0.148489
min,0.000000e+00,22.000000,84.000000,20000.000000,0.050000,0.001000,12322.000000,4.000000,4.000000,0.300000
25%,4.094305e+05,31.000000,192.000000,49230.750000,0.050000,0.001000,36084.750000,6.000000,6.000000,0.300000
50%,2.164241e+06,40.000000,312.000000,76068.500000,0.150000,0.001000,49182.000000,7.000000,7.000000,0.500000
75%,8.026924e+06,50.000000,420.000000,122763.250000,0.150000,0.004000,72198.750000,9.000000,9.000000,0.500000
max,2.578207e+08,59.000000,528.000000,495999.000000,0.300000,0.008000,328587.000000,23.000000,23.000000,0.700000


In [27]:
(df_user['wealth']>5e7).sum()

np.int64(627)

## STATE VECTOR INITIALIZATION

In [13]:
import numpy as np

def initialize_state(user_df, market_df):
    """
    Initialize the RL environment state.

    Args:
        user_df (pd.DataFrame): dataframe containing user profiles
        market_df (pd.DataFrame): processed market data

    Returns:
        state (dict): initial state
    """

    # -------- SAMPLE USER --------
    user = user_df.sample(1).iloc[0]

    # -------- RANDOM START TIME (CIRCULAR) --------
    start_idx = np.random.randint(0, len(market_df))
    market_row = market_df.iloc[start_idx]

    # -------- INITIAL PORTFOLIO --------
    emergency_buffer_months = user["emergency_buffer"]
    monthly_expense = user["expense_base"]
    wealth = user["wealth"]

    if wealth <= 0:
        w_cash = 1.0
        w_eq = 0.0
        w_debt = 0.0
    else:
        # Cash allocation based on emergency buffer
        w_cash = (emergency_buffer_months * monthly_expense) / wealth
        w_cash = min(w_cash, 1.0)

        # Remaining allocation
        remaining = 1 - w_cash

        # Split remaining between equity and debt
        w_eq = user["risk_appetite"] * remaining
        w_debt = remaining - w_eq

    # -------- STATE CONSTRUCTION --------
    state = {
        # Portfolio
        "wealth": wealth,
        "w_eq": w_eq,
        "w_debt": w_debt,
        "w_cash": w_cash,

        # Market
        "volatility": market_row["volatility"],
        "drawdown": market_row["drawdown"],
        "market_indicator": market_row["market_indicator"],

        # Household
        "income": user["income"],
        "income_var": user["income_var"],
        "job_loss_prob": user["job_loss_prob"],
        "expense_base": user["expense_base"],
        "expense": user["expense_base"],
        "emergency_buffer_months": emergency_buffer_months,
        "emergency_buffer_months_target": user["emergency_buffer_target"],
        "risk_appetite": user["risk_appetite"],

        # Time
        "age": user["age"],
        "goal": user["goal"],

        # Internal trackers
        "timestep": start_idx
    }

    return state

In [ ]:
import numpy as np

def action_softmax(logits):
    """
    Convert raw network outputs into valid portfolio weights

    Args:
        logits (np.array): raw outputs from policy network

    Returns:
        weights (np.array): normalized allocation (sum = 1)
    """

    # Stability trick
    logits = logits - np.max(logits)

    exp_vals = np.exp(logits)
    weights = exp_vals / np.sum(exp_vals)

    return weights

## Data Loading and Preprocessing

## Model User Profile

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cols = ["income", "expense_base", "wealth", "emergency_buffer"]

for col in cols:
    plt.figure()
    plt.hist(df[col], bins=100)
    plt.title(f"{col} distribution")
    plt.xlabel(col)
    plt.ylabel("count")
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(df["age"], df["income"], alpha=0.2)
plt.xlabel("Age")
plt.ylabel("Income")
plt.title("Income vs Age")
plt.show()

In [ ]:
plt.figure()
plt.scatter(df["age"], df["wealth"], alpha=0.2)
plt.xlabel("Age")
plt.ylabel("Wealth")
plt.title("Wealth vs Age")
plt.show()

In [ ]:
import pandas as pd

df["age_bin"] = pd.cut(df["age"], bins=range(20, 65, 5))

grouped = df.groupby("age_bin").median(numeric_only=True)

fig, ax1 = plt.subplots()

ax1.plot(grouped.index.astype(str), grouped["income"], color="blue", label="Income")
ax1.set_ylabel("Income", color="blue")

ax2 = ax1.twinx()
ax2.plot(grouped.index.astype(str), grouped["wealth"], color="orange", label="Wealth")
ax2.set_ylabel("Wealth", color="orange")

plt.xticks(rotation=45)
plt.title("Median Income & Wealth by Age Group")
plt.show()

In [ ]:
plt.figure()
plt.scatter(df["age"], df["expense_base"], alpha=0.2)
plt.xlabel("Age")
plt.ylabel("Expenses")
plt.title("Expenses vs Age")
plt.show()

In [ ]:
ratio = df["wealth"] / (df["income"] * 12)

plt.figure()
plt.scatter(df["age"], ratio, alpha=0.2)
plt.xlabel("Age")
plt.ylabel("Wealth / Annual Income")
plt.title("Wealth Ratio vs Age")
plt.show()

In [ ]:
plt.figure()
df.boxplot(column="wealth", by="age_bin", rot=45)
plt.title("Wealth by Age Group")
plt.suptitle("")
plt.show()

## Environment Modelling

In [ ]:
import numpy as np
import pandas as pd


# -----------------------------
# LOAD USER PROFILE
# -----------------------------
def sample_user_from_csv(path, idx):
    df = pd.read_csv(path)
    user = df.sample(1).iloc[idx].to_dict()
    return user


# -----------------------------
# LOAD MARKET DATA
# -----------------------------
def load_market_data(path):
    df = pd.read_csv(path)
    # df["Date"] = pd.to_datetime(df["Date"])
    df = df.reset_index(drop=True)
    return df

In [ ]:
# -----------------------------
# STATE → VECTOR
# -----------------------------
def state_to_vector(state):
    return np.array([
        state["wealth"],
        state["w_eq"],
        state["w_debt"],
        state["w_cash"],
        state["equity_return"],
        state["debt_return"],
        state["cash_return"],
        state["volatility"],
        state["drawdown"],
        state["income"],
        state["income_vol"],
        state["job_loss_prob"],
        state["expenses"],
        state["age"],
        state["goal"]
    ], dtype=np.float32)

In [ ]:
class MarketSimulator:
    """
    Handles market data and circular sampling for episodes.
    """

    def __init__(self, market_returns_data_path):
        self.df = pd.read_csv(market_returns_data_path)

        # Convert Date
        self.df['Date'] = pd.to_datetime(self.df['Date'])

        self.length = len(self.df)

        # Extract numpy arrays
        self.equity_returns = self.df['equity_return'].values
        self.debt_returns = self.df['debt_return'].values
        self.cash_returns = self.df['cash_return'].values
        self.volatility = self.df['volatility'].values
        self.drawdown = self.df['drawdown'].values

    def reset(self):
        """
        Pick random starting index
        """
        self.start_idx = np.random.randint(0, self.length)
        self.t = 0

    def step(self):
        """
        Get next timestep data using circular logic
        """
        idx = (self.start_idx + self.t) % self.length

        state = {
            "equity_return": self.equity_returns[idx],
            "debt_return": self.debt_returns[idx],
            "cash_return": self.cash_returns[idx],
            "volatility": self.volatility[idx],
            "drawdown": self.drawdown[idx]
        }

        self.t += 1

        return state

In [ ]:
import numpy as np


class PortfolioEnv:
    """
    RL Environment for portfolio allocation
    """

    def __init__(self, market_returns_data_path, user_profile):
        self.market = MarketSimulator(market_returns_data_path)

        self.user = user_profile

    def reset(self):
        self.market.reset()

        self.t = 0
        self.initial_wealth = user['wealth_0']

        # initial allocation
        cash_amount = user.emergency_buffer_0 * user.expense_base
        cash_weight = cash_amount/user.wealth_0
        wealth_invest_weight = 1 - cash_weight
        equity_weight = user.risk_appetite*wealth_invest_weight
        debt_weight = 1-cash_weight - equity_weight

        self.weights = np.array([equity_weight, debt_weight, cash_weight])

        return self._get_state()

    def _get_state(self):
        market_state = self.market.step()

        state = np.array([
            self.wealth,
            market_state["equity_return"],
            market_state["debt_return"],
            market_state["cash_return"],
            market_state["volatility"],
            market_state["drawdown"]
        ], dtype=np.float32)

        return state

    def step(self, action):
        """
        action = [w_eq, w_debt, w_cash]
        """

        # Normalize action
        action = np.clip(action, 0, 1)
        action = action / np.sum(action)

        market_state = self.market.step()

        returns = np.array([
            market_state["equity_return"],
            market_state["debt_return"],
            market_state["cash_return"]
        ]) / 100  # convert % → decimal

        # Portfolio return
        portfolio_return = np.dot(action, returns)

        new_wealth = self.wealth * (1 + portfolio_return)

        # Reward = profit
        reward = new_wealth - self.wealth

        self.wealth = new_wealth
        self.weights = action
        self.t += 1

        done = self.t >= self.episode_length

        next_state = np.array([
            self.wealth,
            market_state["equity_return"],
            market_state["debt_return"],
            market_state["cash_return"],
            market_state["volatility"],
            market_state["drawdown"]
        ], dtype=np.float32)

        return next_state, reward, done, {}

In [ ]:
from src.environment.portfolio_env import PortfolioEnv
import numpy as np


env = PortfolioEnv("data/processed/market_data.csv")

state = env.reset()

for t in range(10):
    action = np.random.rand(3)

    next_state, reward, done, _ = env.step(action)

    print(f"Step {t}")
    print("Wealth:", next_state[0])
    print("Reward:", reward)
    print("-" * 30)